In [13]:
import scanpy as sc
import pandas as pd

print("Step 1: Loading the heavy sparse matrix... (this takes a minute)")
# Read the raw expression matrix numbers
adata = sc.read_mtx("count_matrix_sparse.mtx")

print("Step 2: Reading gene names and cell barcodes...")
# Read the list of human genes and the individual cell IDs
genes = pd.read_csv("count_matrix_genes.tsv", header=None, sep="\t")[0].values
barcodes = pd.read_csv("count_matrix_barcodes.tsv", header=None, sep="\t")[0].values

# Double-check data orientation (Scanpy likes cells as rows, genes as columns)
if adata.shape == (len(genes), len(barcodes)):
    print("Orienting matrix correctly...")
    adata = adata.T

# Assign the names to the columns and rows
adata.var_names = genes
adata.obs_names = barcodes

print("Step 3: Merging clinical metadata...")
# Load the patient and clinical data (subtypes, cell types, etc.)
metadata = pd.read_csv("metadata.csv", index_col=0)

# Securely attach the metadata to your single-cell object
adata.obs = adata.obs.join(metadata)

print("\n🎉 SUCCESS! Here is your real breast cancer single-cell setup:")
print(adata)

Step 1: Loading the heavy sparse matrix... (this takes a minute)


ValueError: Line 1: Not a Matrix Market file. Missing banner.

In [14]:
import gzip

print("🔍 DIAGNOSTIC 1: Trying to read as a plain text file...")
try:
    with open("count_matrix_sparse.mtx", "r", encoding="utf-8", errors="ignore") as f:
        for i in range(3):
            print(f"Line {i+1}: {repr(f.readline())}")
except Exception as e:
    print(f"Could not read as text: {e}")

print("\n🔍 DIAGNOSTIC 2: Trying to read as a compressed GZIP file...")
try:
    with gzip.open("count_matrix_sparse.mtx", "rt") as f:
        for i in range(3):
            print(f"Line {i+1}: {repr(f.readline())}")
except Exception as e:
    print(f"Could not read as gzip: {e}")

🔍 DIAGNOSTIC 1: Trying to read as a plain text file...
Could not read as text: [Errno 2] No such file or directory: 'count_matrix_sparse.mtx'

🔍 DIAGNOSTIC 2: Trying to read as a compressed GZIP file...
Could not read as gzip: [Errno 2] No such file or directory: 'count_matrix_sparse.mtx'


In [15]:
import scanpy as sc
import pandas as pd
import os

# 1. Your exact verified folder path
data_dir = r"Z:\Breast Cancer Hits Project\Wu_etal_2021_BRCA_scRNASeq"

# 2. Automatically map to each specific file
matrix_path = os.path.join(data_dir, "count_matrix_sparse.mtx")
genes_path = os.path.join(data_dir, "count_matrix_genes.tsv")
barcodes_path = os.path.join(data_dir, "count_matrix_barcodes.tsv")
metadata_path = os.path.join(data_dir, "metadata.csv")

print("🔍 Verification Step: Checking file connections...")
files = [matrix_path, genes_path, barcodes_path, metadata_path]
all_exist = True
for f in files:
    if os.path.exists(f):
        print(f"✅ Found: {os.path.basename(f)}")
    else:
        print(f"❌ Missing: {os.path.basename(f)}")
        all_exist = False

if all_exist:
    print("\n🚀 All files connected! Starting the matrix import... (this will take 1-2 minutes)")
    try:
        # Load the raw expression numbers
        adata = sc.read_mtx(matrix_path)

        print("🧬 Reading gene names and cell barcodes...")
        genes = pd.read_csv(genes_path, header=None, sep="\t")[0].values
        barcodes = pd.read_csv(barcodes_path, header=None, sep="\t")[0].values

        # Orient matrix if needed (cells as rows, genes as columns)
        if adata.shape == (len(genes), len(barcodes)):
            adata = adata.T

        adata.var_names = genes
        adata.obs_names = barcodes

        print("📊 Merging clinical metadata matrix...")
        metadata = pd.read_csv(metadata_path, index_col=0)
        adata.obs = adata.obs.join(metadata)

        print("\n🎉 SUCCESS! Your breast cancer single-cell object is completely loaded:")
        print(adata)
        
    except Exception as e:
        print(f"\n⚠️ The file path worked, but an import error occurred: {e}")
else:
    print("\n❌ Halt: Please fix the missing files or directory path before proceeding.")

🔍 Verification Step: Checking file connections...
✅ Found: count_matrix_sparse.mtx
✅ Found: count_matrix_genes.tsv
✅ Found: count_matrix_barcodes.tsv
✅ Found: metadata.csv

🚀 All files connected! Starting the matrix import... (this will take 1-2 minutes)
🧬 Reading gene names and cell barcodes...
📊 Merging clinical metadata matrix...

🎉 SUCCESS! Your breast cancer single-cell object is completely loaded:
AnnData object with n_obs × n_vars = 100064 × 29733
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major'


In [16]:
# 1. Look at the first 5 rows of our cell database to see the column names
print("--- METADATA COLUMNS AVAILABLE ---")
display(adata.obs.head())

# 2. Let's see what major cell types are present in this dataset
print("\n--- CELL TYPE COUNTS ---")
# The column name for cell types in this specific paper is usually 'celltype_major' or 'cell_type'
for col in ['celltype_major', 'cell_type', 'CellType']:
    if col in adata.obs.columns:
        print(adata.obs[col].value_counts())
        break
else:
    print("Could not find a direct cell type column. Here are all columns to choose from:")
    print(adata.obs.columns.tolist())

--- METADATA COLUMNS AVAILABLE ---


,orig.ident,nCount_RNA,nFeature_RNA,percent.mito,subtype,celltype_subset,celltype_minor,celltype_major
CID3586_AAGACCTCAGCATGAG,CID3586,4581,1689,1.506221,HER2+,Endothelial ACKR1,Endothelial ACKR1,Endothelial
CID3586_AAGGTTCGTAGTACCT,CID3586,1726,779,5.793743,HER2+,Endothelial ACKR1,Endothelial ACKR1,Endothelial
CID3586_ACCAGTAGTTGTGGCC,CID3586,1229,514,1.383238,HER2+,Endothelial ACKR1,Endothelial ACKR1,Endothelial
CID3586_ACCCACTAGATGTCGG,CID3586,1352,609,1.923077,HER2+,Endothelial ACKR1,Endothelial ACKR1,Endothelial
CID3586_ACTGATGGTCAACTGT,CID3586,1711,807,13.325541,HER2+,Endothelial ACKR1,Endothelial ACKR1,Endothelial



--- CELL TYPE COUNTS ---
celltype_major
T-cells              35214
Cancer Epithelial    24489
Myeloid               9675
Endothelial           7605
CAFs                  6573
PVL                   5423
Normal Epithelial     4355
Plasmablasts          3524
B-cells               3206
Name: count, dtype: int64


In [17]:
# 1. Cleanly isolate ONLY the actual malignant cells
adata_cancer = adata[adata.obs['celltype_major'] == 'Cancer Epithelial'].copy()

print("--- MALIGNANT COHORT ISOLATED ---")
print(f"Total malignant cells remaining: {adata_cancer.n_obs}")

# 2. Check the clinical breast cancer subtypes represented across these cells
print("\n--- CLINICAL SUBTYPE DISTRIBUTION ---")
print(adata_cancer.obs['subtype'].value_counts())

--- MALIGNANT COHORT ISOLATED ---
Total malignant cells remaining: 24489

--- CLINICAL SUBTYPE DISTRIBUTION ---
subtype
ER+      11878
TNBC     10836
HER2+     1775
Name: count, dtype: int64


In [18]:
# 1. Run the differential expression analysis across the clinical subtypes
sc.tl.rank_genes_groups(adata_cancer, 'subtype', method='wilcoxon')

# 2. Extract and format the top 50 upregulated target genes for each subtype
result_cancer = adata_cancer.uns['rank_genes_groups']
subtypes_found = result_cancer['names'].dtype.names
all_subtype_targets = pd.DataFrame({sb: result_cancer['names'][sb] for sb in subtypes_found})

print("\n🎯 TOP GENE TARGETS READY FOR PHARMACOGENETIC ANALYSIS:")
for sb in subtypes_found:
    print(f"\n👉 Top 50 Genes to Target for {sb} Breast Cancer:")
    print(all_subtype_targets[sb].head(50).tolist())


🎯 TOP GENE TARGETS READY FOR PHARMACOGENETIC ANALYSIS:

👉 Top 50 Genes to Target for ER+ Breast Cancer:
['AGR3', 'ANKRD30A', 'AGR2', 'BTG2', 'XIST', 'ESR1', 'PIP', 'GATA3', 'C1orf64', 'GPRC5A', 'SLC39A6', 'SMIM22', 'MLPH', 'XBP1', 'CLU', 'CRABP2', 'SH3BGRL', 'IL6ST', 'SCGB2A2', 'ZG16B', 'ADIRF', 'MAGED2', 'GLUL', 'CST3', 'AZGP1', 'CRIP1', 'TBC1D9', 'BCAM', 'CA12', 'SCGB1D2', 'TRPS1', 'TSPAN13', 'PYCARD', 'SMIM14', 'RAB11FIP1', 'MUC1', 'PBX1', 'GADD45B', 'TXNIP', 'DEGS2', 'TNNT1', 'STC2', 'MALAT1', 'DHRS2', 'SCCPDH', 'ELF3', 'EVL', 'SELM', 'SCGB3A1', 'FAM174A']

👉 Top 50 Genes to Target for HER2+ Breast Cancer:
['ERBB2', 'TXN', 'MIEN1', 'DBI', 'SAT1', 'GRB7', 'LMTK3', 'KRT7', 'TNFSF10', 'SEPP1', 'ZNF706', 'STARD3', 'ALDH2', 'TM7SF2', 'SQLE', 'MUCL1', 'ACADM', 'PABPC1', 'UQCR10', 'MRPL13', 'RPL35', 'TKT', 'COX6B1', 'PDLIM3', 'ETFA', 'SERHL2', 'IDH2', 'MAL2', 'ACSL1', 'FXYD3', 'SMS', 'CTNNBIP1', 'S100P', 'NUPR1', 'PGAP3', 'ABRACL', 'MIF', 'ATP5L', 'COMTD1', 'POLR2K', 'REEP3', 'SEC61B', '

C:\Users\Dell\anaconda3\envs\bca_single_cell\lib\site-packages\scanpy\tools\_rank_genes_groups.py:458: RuntimeWarning: overflow encountered in expm1
  foldchanges = (self.expm1_func(mean_group) + 1e-9) / (


In [19]:
# Fix the warning by logging the data mathematically
sc.pp.log1p(adata_cancer)

# Re-run the ranking script on the properly scaled data
sc.tl.rank_genes_groups(adata_cancer, 'subtype', method='wilcoxon')
result_cancer = adata_cancer.uns['rank_genes_groups']
all_subtype_targets = pd.DataFrame({sb: result_cancer['names'][sb] for sb in subtypes_found})

print("✅ Data logarithmized! Clean signatures are ready.")

✅ Data logarithmized! Clean signatures are ready.
